# How a query runs: the order of the clauses

Practice notebook for the datafromzero SQL lesson **How a query runs: the order of the clauses**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. FROM: start with the whole table

In [ ]:
con.sql("""
SELECT * FROM products
""").show()

### Step 2. WHERE chooses rows

In [ ]:
con.sql("""
SELECT * FROM products WHERE unit_price_usd < 5
""").show()

### Step 3. SELECT chooses columns

In [ ]:
con.sql("""
SELECT category FROM products WHERE unit_price_usd < 5
""").show()

### Step 4. GROUP BY turns rows into groups

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS items
FROM products
WHERE unit_price_usd < 5
GROUP BY category
""").show()

### Step 5. An aggregate cannot go in WHERE

This cell gives an error on purpose. Read the message, then run the next cell.

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS items
FROM products
WHERE unit_price_usd < 5 AND COUNT(*) >= 3
GROUP BY category
""").show()

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS items
FROM products
WHERE unit_price_usd < 5
GROUP BY category
HAVING COUNT(*) >= 3
""").show()

### Step 6. ORDER BY and LIMIT run last

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS items
FROM products
WHERE unit_price_usd < 5
GROUP BY category
HAVING COUNT(*) >= 3
ORDER BY items DESC, category
LIMIT 2
""").show()

### Step 7. What about an alias in WHERE?

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd * 100 AS cents
FROM products
WHERE cents < 400
""").show()

### Step 8. Verify

## Exercise 1. Categories with at least three cheap items

The first lines are given. Add the line that makes the groups and the line that keeps only the groups of three or more. The cell gives an error until you do.

In [ ]:
con.sql("""
SELECT category, COUNT(*) AS items
FROM products
WHERE unit_price_usd <= 4
""").show()

## Exercise 2. Stores with more than 400 completed orders

Show `store_id` and the count, biggest first. Before you run anything, write one sentence in the first cell: which clause does each job (completed orders only, a count per store, stores over 400, the sort). Then write the query.

In [ ]:
# My expected result, in words:
#

In [ ]:
# Your query:

## Exercise 3. Spot the flaw

A colleague wants the products over \$4 shown in cents. The query runs without an error but returns no rows, and nine products cost more than \$4. Find the flaw, then write a query that works.

In [ ]:
con.sql("""
SELECT product_name, unit_price_usd * 100 AS unit_price_usd
FROM products
WHERE unit_price_usd > 400
""").show()

In [ ]:
# Your corrected query: